# Dataset checks and network connectivity

Audit the supplied **80 training and 10 student-test trips**. Read only the explicit `train` and `student_test` directories; no reserved evaluator partition is read. Retain trips 33 and 54 in this raw-data audit and mark their user-requested active-training exclusions.

Save real file inventory, coordinate/time checks and weak-component statistics in `outputs/inventory`. Invalid coordinates include missing, non-finite or out-of-range longitude/latitude. Connectivity uses valid integer node IDs; a node with no incident arc is isolated.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components

root = Path("Student folder")
output = Path("outputs/inventory")
output.mkdir(parents=True, exist_ok=True)
excluded = set(json.loads(Path("outputs/training_exclusions.json").read_text())["excluded_trip_ids"])
expected = {"nodes": 2, "arcs": 2, "track": 3, "route": 1}
files, trips, networks = [], [], []

## Inspect all four files per trip

Dtypes describe the parsed raw file; nonnumeric cells are counted separately from original missing cells. Arc IDs and node IDs must be finite integers in range. Time-gap summaries use finite observed differences and do not repair ordering.

Connected components are **weak** components of the directed road graph. This measures undirected network fragmentation; it does not prove every directed movement is legal.

In [2]:
def invalid_coordinates(table):
    if table.shape[1] < 2:
        return None
    values = table.iloc[:, :2].to_numpy()
    valid = np.isfinite(values).all(axis=1) & (np.abs(values[:, 0]) <= 180) & (np.abs(values[:, 1]) <= 90)
    return int((~valid).sum())


for partition in ["train", "student_test"]:
    for folder in sorted((root / partition).iterdir()):
        if not folder.is_dir():
            continue
        trip_id = folder.name
        active = partition == "train" and trip_id not in excluded
        tables = {}
        for kind, columns in expected.items():
            path = folder / f"{trip_id}.{kind}"
            error = None
            try:
                raw = pd.read_csv(path, sep=r"\s+", header=None) if path.is_file() else pd.DataFrame()
            except (pd.errors.ParserError, pd.errors.EmptyDataError) as exc:
                raw, error = pd.DataFrame(), str(exc)
            numeric = raw.apply(pd.to_numeric, errors="coerce")
            tables[kind] = numeric
            files.append({"partition": partition, "trip_id": trip_id, "active_training": active,
                "kind": kind, "path": str(path), "exists": path.is_file(),
                "bytes": path.stat().st_size if path.is_file() else None,
                "rows": len(raw), "columns": raw.shape[1], "expected_columns": columns,
                "schema_ok": raw.shape[1] == columns and error is None,
                "dtypes": json.dumps(raw.dtypes.astype(str).tolist()),
                "missing_cells": int(raw.isna().sum().sum()),
                "nonnumeric_cells": int((numeric.isna() & raw.notna()).sum().sum()), "parse_error": error})

        nodes, arcs, track, route = [tables[k] for k in ["nodes", "arcs", "track", "route"]]
        record = {"partition": partition, "trip_id": trip_id, "active_training": active,
                  "excluded_training_trip": partition == "train" and trip_id in excluded,
                  "nodes": len(nodes), "arcs": len(arcs), "gps_points": len(track), "route_segments": len(route),
                  "invalid_node_coordinates": invalid_coordinates(nodes),
                  "invalid_gps_coordinates": invalid_coordinates(track)}
        for name, table in [("network", nodes), ("gps", track)]:
            if table.shape[1] >= 2:
                for axis, column in [("lon", 0), ("lat", 1)]:
                    record[f"{name}_{axis}_min"] = table[column].min()
                    record[f"{name}_{axis}_max"] = table[column].max()
        if track.shape[1] == 3 and len(track):
            time = track[2].to_numpy()
            gaps = np.diff(time)
            finite = gaps[np.isfinite(gaps)]
            record.update(nonfinite_timestamps=int((~np.isfinite(time)).sum()),
                timestamps_strictly_increasing=bool(np.isfinite(time).all() and (gaps > 0).all()),
                non_increasing_steps=int((gaps <= 0).sum()), gaps_gt_5s=int((gaps > 5).sum()),
                duration_s=float(time[-1] - time[0]),
                min_gap_s=float(finite.min()) if len(finite) else None,
                median_gap_s=float(np.median(finite)) if len(finite) else None,
                max_gap_s=float(finite.max()) if len(finite) else None)
        if arcs.shape[1] == 2 and len(nodes):
            values = arcs.to_numpy()
            valid = np.isfinite(values).all(axis=1) & (values >= 0).all(axis=1) & (values < len(nodes)).all(axis=1)
            valid &= (values == np.floor(values)).all(axis=1)
            edges = values[valid].astype(int)
            graph = csr_matrix((np.ones(len(edges)), (edges[:, 0], edges[:, 1])), shape=(len(nodes), len(nodes)))
            count, labels = connected_components(graph, directed=True, connection="weak")
            sizes = np.bincount(labels)
            networks.append({"partition": partition, "trip_id": trip_id, "active_training": active,
                "nodes": len(nodes), "arcs": len(arcs), "invalid_arcs": int((~valid).sum()),
                "weak_components": int(count), "largest_component_nodes": int(sizes.max()),
                "largest_component_fraction": float(sizes.max() / len(nodes)),
                "isolated_nodes": int((np.bincount(edges.ravel(), minlength=len(nodes)) == 0).sum())})
            record["invalid_arcs"] = int((~valid).sum())
        if route.shape[1] == 1:
            values = route[0].to_numpy()
            valid = np.isfinite(values) & (values == np.floor(values)) & (values >= 0) & (values < len(arcs))
            record["invalid_route_arc_ids"] = int((~valid).sum())
        trips.append(record)
    print("Audited:", partition, flush=True)

Audited: train


Audited: student_test


## Save the audit evidence

The official raw training partition remains 80 trips; active training is 78 by user instruction. This audit documents that deviation rather than deleting files or changing existing model folds.

In [3]:
file_inventory = pd.DataFrame(files)
trip_checks = pd.DataFrame(trips)
connectivity = pd.DataFrame(networks)
file_inventory.to_csv(output / "file_inventory.csv", index=False)
trip_checks.to_csv(output / "trip_checks.csv", index=False)
connectivity.to_csv(output / "network_connectivity.csv", index=False)
summary = trip_checks.groupby("partition").agg(trips=("trip_id", "size"), gps_points=("gps_points", "sum"),
    nodes=("nodes", "sum"), arcs=("arcs", "sum"), active_training_trips=("active_training", "sum"))
summary.to_csv(output / "partition_summary.csv")
assert summary.loc["train", "trips"] == 80 and summary.loc["student_test", "trips"] == 10
assert summary.loc["train", "active_training_trips"] == 78
assert len(file_inventory) == 360 and len(connectivity) == 90
assert file_inventory["exists"].all() and file_inventory["schema_ok"].all()
assert file_inventory["nonnumeric_cells"].sum() == 0
print("Checked", len(file_inventory), "files across", len(trip_checks), "trips.")
display(summary)
display(connectivity.sort_values("weak_components", ascending=False).head(10))
display(trip_checks.sort_values("max_gap_s", ascending=False).head(5))

Checked 360 files across 90 trips.


,trips,gps_points,nodes,arcs,active_training_trips
partition,,,,,
student_test,10,19160,90689,185041,0
train,80,208417,1933485,3876904,78


,partition,trip_id,active_training,nodes,arcs,invalid_arcs,weak_components,largest_component_nodes,largest_component_fraction,isolated_nodes
35,train,00000043,True,109070,215620,0,421,106605,0.977400,0
44,train,00000054,False,63822,116916,0,213,61931,0.970371,0
69,train,00000086,True,114868,237656,0,193,112705,0.981170,0
0,train,00000001,True,88809,165790,0,175,86854,0.977986,0
25,train,00000032,True,124540,259437,0,126,123433,0.991111,0
28,train,00000035,True,21126,35212,0,119,19314,0.914229,0
59,train,00000074,True,34654,61055,0,118,33120,0.955734,0
80,student_test,00000000,False,8542,18196,0,100,7466,0.874034,0
14,train,00000019,True,52694,108517,0,90,51144,0.970585,0
18,train,00000024,True,25782,51609,0,84,24840,0.963463,0


,partition,trip_id,active_training,excluded_training_trip,nodes,arcs,gps_points,route_segments,invalid_node_coordinates,invalid_gps_coordinates,...,nonfinite_timestamps,timestamps_strictly_increasing,non_increasing_steps,gaps_gt_5s,duration_s,min_gap_s,median_gap_s,max_gap_s,invalid_arcs,invalid_route_arc_ids
72,train,00000090,True,False,4010,7942,536,235,0,0,...,0,True,0,20,154030.0,1.0,1.0,153064.0,0,0
61,train,00000078,True,False,75230,151458,2909,1765,0,0,...,0,True,0,124,57836.0,1.0,1.0,52810.0,0,0
55,train,00000069,True,False,8556,17193,2067,467,0,0,...,0,True,0,2,5558.0,1.0,1.0,2658.0,0,0
16,train,00000021,True,False,732,1510,15783,277,0,0,...,0,True,0,2,18605.0,1.0,1.0,2161.0,0,0
8,train,00000011,True,False,2649,5393,2828,565,0,0,...,0,True,0,1,3735.0,1.0,1.0,909.0,0,0
